In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from dotenv import load_dotenv
import os
from langgraph.checkpoint.memory import MemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

print(llm)



model='models/gemini-2.5-flash' google_api_key=SecretStr('**********') client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000001A10C9A5A90> default_metadata=() model_kwargs={}


In [16]:
def merge_score_dict(existing: dict, new_update: dict):
    if existing is None:
        return new_update
    return {**existing, **new_update}


class AnalyzerState(TypedDict):
    raw_text: str
    safety_score: Annotated[dict[str, int], merge_score_dict]


# Create Nodes
def toxicity_node(state: AnalyzerState) -> dict:

    print("\n[Branch1] Analyzing teh Toxicity and Hate Speech")

    prompt = (
        "Analyze the following Text for agrression or hate speech or toxicity"
        f"Text:\n{state["raw_text"]}"
    )

    response = llm.invoke(prompt)

    try:
        score = int(response.content.strip())
    except ValueError:
        score = 0

    return {"safety_score": {"toxicity_level": score}}


def copyright_node(state: AnalyzerState) -> dict:
    print("\n[Branch1] Analyzing teh Toxicity and Hate Speech")

    prompt = "Analyze the following Text for Plagarism" f"Text:\n{state["raw_text"]}"

    response = llm.invoke(prompt)

    try:
        score = int(response.content.strip())
    except ValueError:
        score = 0

    return {"copyright_score": {"copyright_score": score}}

In [22]:
builder = StateGraph(AnalyzerState)


# Adding Nodes
builder.add_node("toxity_node", toxicity_node)
builder.add_node("copyright_node", copyright_node)


# Adding Edges
builder.add_edge(START, "toxity_node")
builder.add_edge(START, "copyright_node")
builder.add_edge("toxity_node", END)
builder.add_edge("copyright_node", END)

app = builder.compile()